# ParkCast SF — POI Density Features

For each master block we count OpenStreetMap points of interest within a 200m radius,
bucketed into four categories. Parking demand is driven by *why* people drive there
(restaurants, shops, transit, venues) — these are static features that explain
persistent demand patterns without needing event rows or time-of-day priors.

**Output:** `data/block_poi_features.csv` keyed on `cnn`.

## Imports

In [1]:
import os
import json
import time
import requests
import numpy as np
import pandas as pd
from scipy.spatial import cKDTree

## Configuration

In [2]:
PROJECT_DIR = os.path.dirname(os.getcwd())
DATA_DIR = os.path.join(PROJECT_DIR, "data")
MODEL_DIR = os.path.join(PROJECT_DIR, "app", "models")

MASTER = os.path.join(MODEL_DIR, "master_blocks.parquet")
POI_CACHE = os.path.join(DATA_DIR, "sf_pois.json")
OUT = os.path.join(DATA_DIR, "block_poi_features.csv")

# SF bounding box (south, west, north, east)
BBOX = (37.700, -122.550, 37.830, -122.350)
RADIUS_M = 200
OVERPASS_URL = "https://overpass-api.de/api/interpreter"

# OSM tag → our category.
CATEGORIES = {
    "dining":      [('amenity', ['restaurant', 'cafe', 'bar', 'pub',
                                    'fast_food', 'food_court', 'ice_cream'])],
    "retail":      [('shop', None)],
    "transit":     [('public_transport', ['station']),
                    ('railway', ['station', 'subway_entrance']),
                    ('amenity', ['bus_station'])],
    "attraction":  [('tourism', ['museum', 'attraction', 'hotel', 'gallery']),
                    ('amenity', ['theatre', 'cinema', 'nightclub',
                                    'arts_centre', 'hospital', 'university',
                                    'college', 'library'])],
}

## `fetch_pois()`

Query Overpass for all nodes/ways with any tag in our category set, across the
SF bounding box. Results cached to `data/sf_pois.json` so re-runs don't hit the
API again.

In [3]:
def build_overpass_query():
    s, w, n, e = BBOX
    bbox = f"{s},{w},{n},{e}"
    parts = []
    for _cat, specs in CATEGORIES.items():
        for key, vals in specs:
            if vals is None:
                parts.append(f'node["{key}"]({bbox});')
                parts.append(f'way["{key}"]({bbox});')
            else:
                regex = "|".join(vals)
                parts.append(f'node["{key}"~"^({regex})$"]({bbox});')
                parts.append(f'way["{key}"~"^({regex})$"]({bbox});')
    body = "\n  ".join(parts)
    return f"[out:json][timeout:180];\n(\n  {body}\n);\nout center tags;"

# Overpass public mirrors reject requests that don't identify themselves.
# Without User-Agent some return 406 Not Acceptable; with a contactable
# UA they serve the query normally.
HEADERS = {
    "User-Agent": "ParkCastSF/1.0 (https://github.com/parkcast-sf; contact via github)",
    "Accept": "application/json",
}

def fetch_pois():
    if os.path.exists(POI_CACHE):
        print(f"Loading cached POIs from {POI_CACHE}")
        with open(POI_CACHE) as f:
            return json.load(f)
    q = build_overpass_query()
    print(f"Querying Overpass ({len(q):,} char query)...")
    r = requests.post(OVERPASS_URL, data={"data": q},
                      headers=HEADERS, timeout=240)
    r.raise_for_status()
    payload = r.json()
    with open(POI_CACHE, "w") as f:
        json.dump(payload, f)
    print(f"  Cached {len(payload.get('elements', [])):,} elements → {POI_CACHE}")
    return payload

## `categorize_element()`

OSM elements can carry multiple tags. A "restaurant inside a hotel" has both
`amenity=restaurant` and `tourism=hotel`. We assign it to the first category
whose tag matches — order in CATEGORIES is the priority: dining > retail >
transit > attraction.

In [4]:
def categorize_element(tags):
    for cat, specs in CATEGORIES.items():
        for key, vals in specs:
            if key not in tags:
                continue
            if vals is None or tags[key] in vals:
                return cat
    return None

def elements_to_df(payload):
    rows = []
    for el in payload.get("elements", []):
        tags = el.get("tags", {}) or {}
        cat = categorize_element(tags)
        if cat is None:
            continue
        if el["type"] == "node":
            lat, lon = el.get("lat"), el.get("lon")
        else:
            c = el.get("center") or {}
            lat, lon = c.get("lat"), c.get("lon")
        if lat is None or lon is None:
            continue
        rows.append((cat, float(lat), float(lon)))
    return pd.DataFrame(rows, columns=["category", "lat", "lon"])

## `count_within_radius()`

For every master block, count POIs within `RADIUS_M` meters per category.

We project (lat, lon) to local-metric x/y using the mean latitude of SF
(~37.77°). 1° latitude ≈ 111 km; 1° longitude ≈ 111 * cos(lat) km. This lets
us use a Euclidean KDTree with a radius in meters — no haversine per pair.

In [5]:
LAT0 = 37.77
M_PER_DEG_LAT = 111_000.0
M_PER_DEG_LON = 111_000.0 * np.cos(np.radians(LAT0))

def to_xy(lat, lon):
    y = (lat - LAT0) * M_PER_DEG_LAT
    x = (lon + 122.4194) * M_PER_DEG_LON
    return x, y

def count_within_radius(blocks, pois, radius_m):
    bx, by = to_xy(blocks["lat"].values, blocks["lon"].values)
    block_xy = np.column_stack([bx, by])
    out = pd.DataFrame({"cnn": blocks["cnn"].values})
    for cat in CATEGORIES:
        sub = pois[pois["category"] == cat]
        if sub.empty:
            out[f"poi_{cat}_{radius_m}m"] = 0
            continue
        px, py = to_xy(sub["lat"].values, sub["lon"].values)
        tree = cKDTree(np.column_stack([px, py]))
        counts = tree.query_ball_point(block_xy, r=radius_m)
        out[f"poi_{cat}_{radius_m}m"] = [len(c) for c in counts]
    return out

## Pipeline

In [6]:
print("=" * 60)
print("ParkCast SF — POI Density Features")
print("=" * 60)

master = pd.read_parquet(MASTER)
print(f"Master blocks: {len(master):,}")

payload = fetch_pois()
pois = elements_to_df(payload)
print(f"Tagged POIs: {len(pois):,}")
print("  Per category:")
print(pois["category"].value_counts().to_string())

feats = count_within_radius(master, pois, RADIUS_M)
feats.to_csv(OUT, index=False)

print(f"\nSaved → {OUT}")
print("  Summary per feature:")
print(feats.drop(columns=["cnn"]).describe().round(1).to_string())
print("=" * 60)

ParkCast SF — POI Density Features
Master blocks: 12,242
Querying Overpass (1,189 char query)...


  Cached 9,112 elements → /Users/kayvan/Desktop/MSDS/ml-ops/Project/ml-ops-final-project-team-ParkCast-SF/data/sf_pois.json
Tagged POIs: 9,112
  Per category:
category
retail        4438
dining        3720
attraction     796
transit        158

Saved → /Users/kayvan/Desktop/MSDS/ml-ops/Project/ml-ops-final-project-team-ParkCast-SF/data/block_poi_features.csv
  Summary per feature:
       poi_dining_200m  poi_retail_200m  poi_transit_200m  poi_attraction_200m
count          12242.0          12242.0           12242.0              12242.0
mean               6.4              7.6               0.3                  1.2
std               12.1             15.9               1.0                  3.4
min                0.0              0.0               0.0                  0.0
25%                0.0              0.0               0.0                  0.0
50%                1.0              2.0               0.0                  0.0
75%                7.0              8.0               0.0      